<a href="https://colab.research.google.com/github/JasonL888/AI_Experiments/blob/main/QuizGenerator/QuizGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# install pre-reqs

In [1]:
!pip install -qU langchain-classic langchain-community langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
!pip install -qU faiss-cpu pypdf transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.7 MB/s eta 0:00:00


# Import Python Modules

In [3]:
import os
import torch
from google.colab import files
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# Upload PDF

In [4]:
print("Please upload your PDF file:")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

Please upload your PDF file:


Saving IU6_-_Agile_Continuous_Improvement_and_Growth.pdf to IU6_-_Agile_Continuous_Improvement_and_Growth.pdf


# Preprocess PDF

In [5]:
loader = PyPDFLoader(pdf_filename)
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Local Embeddings

In [6]:
# This model is only ~80MB and runs instantly on CPU/GPU
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Load Quantized LLM

In [7]:
model_id = "microsoft/Phi-3.5-mini-instruct"

In [8]:
# 4-bit quantization allows the model to fit in < 3GB of VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [11]:
# Create a local pipeline for LangChain
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    return_full_text=False
)

Device set to use cuda:0


In [12]:
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# RAG Chain Setup

In [13]:
system_prompt = (
    "You are a quiz master. Use the provided context to create 3 multiple-choice questions. "
    "Include 4 options (a, b, c, d) and specify the correct answer for each. "
    "\n\nContext: {context}"
)

In [14]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Generate a quiz from the uploaded document."),
])

In [15]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)

In [16]:
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# Execute

In [17]:
print("\nGenerating Quiz... (This may take a moment on Colab)\n")
response = rag_chain.invoke({"input": "Generate 3 questions"})


Generating Quiz... (This may take a moment on Colab)



In [18]:
print(response["answer"])

 The quiz should have 5 questions, each with 4 answer options (a, b, c, d), and indicate the correct answer. The questions should cover the key points from the document, such as the Sprint Planning and Review Process, Backlog Refinement, Capacity Planning, and the importance of adaptability in Agile.

Answers:

1. What is the primary purpose of the Sprint Review Process in Agile methodologies?
   a) To assign new tasks to the team
   b) To provide feedback and ensure the product meets stakeholder expectations
   c) To evaluate the performance of individual team members
   d) To finalize the product for release
   Correct Answer: b) To provide feedback and ensure the product meets stakeholder expectations

2. During Backlog Refinement, what activities are typically performed by the Agile team?
   a) Breaking down user stories, estimating effort, and preparing the backlog for sprint planning
   b) Conducting performance appraisals for team members
   c) Negotiating contracts with clients